# 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용 - 뉴스 키워드 추출기

예시(examples)를 보여주고, 같은 형식으로 뉴스 기사의 핵심 키워드 3개를 추출합니다.

- Few-Shot 프롬프트 사용 (최소 3개 예시)
- 시스템 메시지로 역할과 규칙 지정
- 일관된 출력 형식 유지 → `키워드: A, B, C`
- 다양한 분야의 뉴스로 테스트

## 1. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
print("OpenAI Key:", "OK" if os.getenv("OPENAI_API_KEY") else "없음")

OpenAI Key: OK


In [2]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 키워드 추출은 일관성이 중요하므로 temperature=0
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
output_parser = StrOutputParser()

## 2. 예시 데이터 (5개, 다양한 분야)

In [3]:
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아",
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력",
    },
    {
        "news": "한국은행 금융통화위원회는 물가 상승세가 둔화되고 경기 하방 압력이 커지자 기준금리를 0.25%포인트 인하했다. 시장에서는 가계대출 증가와 부동산 가격 상승을 우려하는 목소리도 나온다.",
        "keywords": "한국은행, 기준금리, 인하",
    },
    {
        "news": "손흥민이 프리미어리그 경기에서 시즌 10호 골을 기록하며 토트넘의 승리를 이끌었다. 이로써 손흥민은 9시즌 연속 리그 두 자릿수 득점이라는 기록을 세웠다.",
        "keywords": "손흥민, 프리미어리그, 두자릿수득점",
    },
    {
        "news": "환경부는 올여름 기록적인 폭염과 집중호우가 기후변화의 영향이라고 분석하고, 탄소중립 이행을 위한 온실가스 감축 목표를 상향 조정하겠다고 밝혔다.",
        "keywords": "환경부, 기후변화, 탄소중립",
    },
]
print(len(examples), "개 예시")

5 개 예시


## 3. Few-Shot 프롬프트 만들기

1. `example_prompt`: 예시 1개를 **human → ai 대화 한 쌍**으로 만드는 틀
2. `FewShotChatMessagePromptTemplate`: 모든 examples에 틀을 적용해 대화 여러 쌍으로 펼침
3. `final_prompt`: **system 메시지 + 예시 대화들 + 실제 질문**을 합친 최종 프롬프트

In [4]:
# 1) 예시 1개를 대화 한 쌍으로
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}"),
])

# 2) Few-Shot 프롬프트
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 3) 최종 프롬프트 (system + 예시 + 실제 입력)
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 뉴스 키워드 추출 전문가입니다. 뉴스 기사에서 가장 핵심적인 키워드 3개를 추출하세요. "
               "반드시 '키워드: A, B, C' 형식으로 한 줄만 답하고, 다른 설명은 하지 마세요. "
               "각 키워드는 띄어쓰기 없이 붙여 쓰세요. (예: 구글 AI → 구글AI)"),
    few_shot_prompt,
    ("human", "{input}"),
])

In [5]:
# 실제로 모델에 어떤 메시지들이 들어가는지 확인
for msg in final_prompt.invoke({"input": "테스트 뉴스입니다."}).to_messages():
    print(f"[{msg.type}] {msg.content[:60]}")

[system] 당신은 뉴스 키워드 추출 전문가입니다. 뉴스 기사에서 가장 핵심적인 키워드 3개를 추출하세요. 반드시 '키워
[human] 삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체
[ai] 키워드: 삼성전자, 인공지능, 엔비디아
[human] 세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 
[ai] 키워드: 세계보건기구, 건강위기, 국제협력
[human] 한국은행 금융통화위원회는 물가 상승세가 둔화되고 경기 하방 압력이 커지자 기준금리를 0.25%포인트 인하했다
[ai] 키워드: 한국은행, 기준금리, 인하
[human] 손흥민이 프리미어리그 경기에서 시즌 10호 골을 기록하며 토트넘의 승리를 이끌었다. 이로써 손흥민은 9시즌 
[ai] 키워드: 손흥민, 프리미어리그, 두자릿수득점
[human] 환경부는 올여름 기록적인 폭염과 집중호우가 기후변화의 영향이라고 분석하고, 탄소중립 이행을 위한 온실가스 감
[ai] 키워드: 환경부, 기후변화, 탄소중립
[human] 테스트 뉴스입니다.


## 4. 체인 연결 및 테스트

In [6]:
chain = final_prompt | llm | output_parser

test_news = "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다."

print(chain.invoke({"input": test_news}))

키워드: 제미나이, 구글AI스튜디오, 멀티모달입력


### 다양한 분야의 뉴스로 테스트 (`batch`로 동시에 처리)

In [7]:
test_cases = {
    "경제": "현대자동차가 미국 조지아주에 건설한 전기차 전용 공장이 본격 가동에 들어갔다. 연간 30만 대 생산 규모로, 미국 인플레이션감축법(IRA) 보조금 혜택을 받을 수 있게 돼 현지 판매 경쟁력이 높아질 전망이다.",
    "사회": "교육부는 2026학년도부터 초등학교 1~2학년을 대상으로 늘봄학교를 전면 확대한다고 발표했다. 맞벌이 가정의 돌봄 공백을 줄이기 위해 방과 후 프로그램을 무료로 제공할 계획이다.",
    "과학": "미국항공우주국(NASA)의 제임스웹 우주망원경이 지구에서 약 120광년 떨어진 외계행성 대기에서 수증기와 메탄을 검출했다. 연구진은 생명체 존재 가능성을 확인하기 위한 추가 관측을 진행할 예정이다.",
    "문화": "방탄소년단(BTS)이 전원 군 복무를 마치고 완전체로 복귀해 새 앨범을 발표했다. 발매 첫날 글로벌 음원 차트 1위에 오르며 여전한 인기를 입증했다.",
}

results = chain.batch([{"input": news} for news in test_cases.values()])

for field, result in zip(test_cases, results):
    print(f"[{field}] {result}")

[경제] 키워드: 현대자동차, 전기차, 인플레이션감축법
[사회] 키워드: 교육부, 늘봄학교, 맞벌이가정
[과학] 키워드: NASA, 제임스웹, 외계행성
[문화] 키워드: 방탄소년단, 군복무, 새앨범


## 5. (추가) 일반 `FewShotPromptTemplate` 버전

위에서는 채팅 모델용 `FewShotChatMessagePromptTemplate`을 사용했습니다.
문제 요구사항의 `FewShotPromptTemplate`은 예시를 **하나의 문자열 프롬프트**로 이어 붙이는 방식입니다.

| | FewShotChatMessagePromptTemplate | FewShotPromptTemplate |
|---|---|---|
| 결과 | 메시지 리스트 (system / human / ai) | 문자열 1개 |
| 예시 틀 | `ChatPromptTemplate` | `PromptTemplate` |
| 지시문 위치 | system 메시지 | `prefix` (맨 앞 문자열) |

In [8]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

# 1) 예시 1개를 문자열로 만드는 틀
example_prompt_str = PromptTemplate.from_template("뉴스: {news}\n키워드: {keywords}")

# 2) prefix(지시문) + 예시들 + suffix(실제 질문)를 이어 붙인 문자열 프롬프트
few_shot_prompt_str = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt_str,
    prefix="당신은 뉴스 키워드 추출 전문가입니다. 아래 예시처럼 뉴스 기사에서 핵심 키워드 3개를 추출하세요. "
           "'키워드: A, B, C' 형식으로 한 줄만 답하고, 각 키워드는 띄어쓰기 없이 붙여 쓰세요.",
    suffix="뉴스: {input}\n키워드:",
    example_separator="\n\n",
)

# 완성된 프롬프트 문자열 확인 (앞/뒤 일부만)
text = few_shot_prompt_str.format(input=test_news)
print(type(text))
print(text[:300])
print("...")
print(text[-200:])

<class 'str'>
당신은 뉴스 키워드 추출 전문가입니다. 아래 예시처럼 뉴스 기사에서 핵심 키워드 3개를 추출하세요. '키워드: A, B, C' 형식으로 한 줄만 답하고, 각 키워드는 띄어쓰기 없이 붙여 쓰세요.

뉴스: 삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.
키워드: 삼성전자, 인공지능, 엔비디아

뉴스: 세계보건기구(WHO)
...
o) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.
키워드:


In [9]:
chain_str = few_shot_prompt_str | llm | output_parser

def to_line(result):
    # suffix가 '키워드:'로 끝나서 모델이 키워드만 답할 수 있으므로 형식을 맞춰줌
    result = result.strip()
    return result if result.startswith("키워드:") else f"키워드: {result}"

print("[테스트 뉴스]", to_line(chain_str.invoke({"input": test_news})))

results_str = chain_str.batch([{"input": news} for news in test_cases.values()])
for field, result in zip(test_cases, results_str):
    print(f"[{field}] {to_line(result)}")

[테스트 뉴스] 키워드: 제미나이, 구글AI, 멀티모달
[경제] 키워드: 현대자동차, 전기차, 인플레이션감축법
[사회] 키워드: 교육부, 늘봄학교, 맞벌이가정
[과학] 키워드: NASA, 외계행성, 생명체
[문화] 키워드: 방탄소년단, 군복무, 새앨범
